# Plotly Geospatial, 3D & Financial Charts: Beginner Guide
Create world maps, interactive 3D charts, OHLC financial candlesticks, and conversion funnel workflows.

### 📚 What You Will Learn in this Guide:
- **Geospatial World Mapping (`px.scatter_geo`, `px.choropleth`)**: How to plot data on interactive world and country maps.
- **Interactive 3D Web Graphics (`px.scatter_3d`, `go.Surface`)**: How to build rotatable 3D charts in Python.
- **Financial Candlestick Charts (`go.Candlestick`)**: How to plot trading Open-High-Low-Close settlement volatility.
- **Conversion Funnels (`px.funnel`)**: How to track user drop-off across checkout stages.

> **💡 Beginner Note**: Every single concept is isolated in its own section with:
> 1. **What is this?** (Plain English explanation)
> 2. **Why do we use it?** (Real-world intuition)
> 3. **Syntax & Parameters** (Parameter-by-parameter breakdown)
> 4. **Live Python Code** with outputs using `data/raw_transactions.csv`.

In [1]:
# Step 1: Import all necessary libraries
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Step 2: Set clean visual defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 10

# Step 3: Load the transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)

# Step 4: Ensure dates and numeric values are clean
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce')
df['account_age_months'] = pd.to_numeric(df['account_age_months'], errors='coerce')
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)
df = df.dropna(subset=['transaction_amount', 'transaction_date']).reset_index(drop=True)

print(f"✅ Successfully loaded {len(df)} transactions from {csv_path}")
df.head(3)

✅ Successfully loaded 14262 transactions from data/raw_transactions.csv


## 🔹 Interactive World Maps: `px.scatter_geo()`

### 1. What is this?
`px.scatter_geo()` places bubbles on an interactive globe map based on country codes.

### 2. Why do we use it?
Use world maps for global international sales, cross-border payments, and multi-country KPIs.

### 3. Syntax & Parameters
```python
fig = px.scatter_geo(geo_df, locations='country_code', size='revenue')
```

In [2]:
mock_geo = pd.DataFrame({
    'country': ['USA', 'GBR', 'DEU', 'FRA', 'JPN', 'CAN', 'AUS'],
    'spend': [520000, 310000, 290000, 240000, 195000, 180000, 160000]
})

fig = px.scatter_geo(mock_geo, locations='country', size='spend', projection='natural earth', title='Global Transaction Volumes (Interactive World Map)')
fig.show()

Figure(Global Transaction Volumes (Interactive World Map))

## 🔹 Colored Country Choropleths: `px.choropleth()`

### 1. What is this?
`px.choropleth()` colors entire country boundaries based on numerical rates (like fraud percentage or market penetration).

### 2. Why do we use it?
Standard for heat-mapping international metrics.

### 3. Syntax & Parameters
```python
fig = px.choropleth(geo_df, locations='country', color='rate', color_continuous_scale='Reds')
```

In [3]:
mock_geo['fraud_rate'] = [0.021, 0.014, 0.011, 0.028, 0.007, 0.015, 0.019]

fig = px.choropleth(mock_geo, locations='country', color='fraud_rate', color_continuous_scale='Reds', title='International Fraud Risk Heat Map (px.choropleth)')
fig.show()

Figure(International Fraud Risk Heat Map (px.choropleth))

## 🔹 Rotatable 3D Scatter: `px.scatter_3d()`

### 1. What is this?
`px.scatter_3d()` renders an interactive 3D chart that you can click and rotate in any direction in your browser.

### 2. Why do we use it?
Great for multi-dimensional data exploration and presentation demos.

### 3. Syntax & Parameters
```python
fig = px.scatter_3d(df, x='age', y='amount', z='fraud', color='card')
```

In [4]:
fig = px.scatter_3d(
    df.head(150), 
    x='account_age_months', 
    y='transaction_amount', 
    z='is_fraud', 
    color='card_type', 
    size='transaction_amount', 
    title='Rotatable 3D Feature Space (Click and drag to rotate!)'
)
fig.show()

Figure(Rotatable 3D Feature Space (Click and drag to rotate!))

## 🔹 Financial Candlesticks: `go.Candlestick()`

### 1. What is this?
Candlestick charts display 4 critical prices for each trading day:
- **Open**: Starting price
- **High**: Peak price
- **Low**: Lowest price
- **Close**: Final settlement price.

### 2. Why do we use it?
Standard chart used by quantitative traders, banks, and cryptocurrency exchanges.

### 3. Syntax & Parameters
```python
fig = go.Figure(data=[go.Candlestick(x=dates, open=o, high=h, low=l, close=c)])
```

In [5]:
daily_ohlc = df.set_index('transaction_date').resample('D')['transaction_amount'].agg(['first', 'max', 'min', 'last']).dropna().head(14)

fig = go.Figure(data=[go.Candlestick(
    x=daily_ohlc.index,
    open=daily_ohlc['first'],
    high=daily_ohlc['max'],
    low=daily_ohlc['min'],
    close=daily_ohlc['last']
)])

fig.update_layout(title='14-Day Settlement Volatility Candlestick Chart', xaxis_rangeslider_visible=False)
fig.show()

Figure(14-Day Settlement Volatility Candlestick Chart)

## 🔹 Conversion Funnels: `px.funnel()`

### 1. What is this?
`px.funnel()` visualizes a multi-step user journey, showing how many users advance or drop off at each sequential stage.

### 2. Why do we use it?
Used by product managers and e-commerce teams to find bottlenecks in the checkout process.

### 3. Syntax & Parameters
```python
fig = px.funnel(funnel_df, x='users_count', y='stage_name')
```

In [6]:
funnel_df = pd.DataFrame({
    'stage': ['1. Cart Opened', '2. Address Filled', '3. Payment Selected', '4. OTP Verified', '5. Transaction Settled'],
    'users': [15000, 12200, 10800, 9900, 9650]
})

fig = px.funnel(funnel_df, x='users', y='stage', title='Checkout Conversion Funnel (Where do users drop off?)')
fig.show()

Figure(Checkout Conversion Funnel (Where do users drop off?))